In [1]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

/home/randy/Documents/ACompArc/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_tinystories_model(model_id, revision, attn_implementation="eager"):
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)

    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, attn_implementation=attn_implementation)

    model.eval()

    return model, tokenizer

In [3]:

model_id = "roneneldan/TinyStories-3M"
revision = "cfaf26ec85ecdfc1bd7c2638104cce55cb67f894"

model, tokenizer = load_tinystories_model(model_id, revision)

W0923 21:48:28.218000 23159 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0923 21:48:28.248000 23159 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 108/108 [00:00<00:00, 29939.51it/s]
[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-3M
Key                                                               | Status     |  | 
------------------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.a

In [4]:
print("Model class:", type(model).__name__)
print("Number of blocks:", len(model.transformer.h))
print("Hidden dimension:", model.config.hidden_size)
print("Attention heads:", model.config.num_heads)
print("Vocabulary size:", model.config.vocab_size)
print("Training mode:", model.training)

Model class: GPTNeoForCausalLM
Number of blocks: 8
Hidden dimension: 128
Attention heads: 16
Vocabulary size: 50257
Training mode: False


In [5]:
print(type(tokenizer))

<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>


In [6]:
PROMPT_LENGTH = 32
GENERATED_LENGTH = 20

prompt = (
    "Once upon a time, a curious little girl named Lily lived in a "
    "quiet village beside a green forest. Every morning, she carried "
    "fresh bread to her grandmother and watched the birds in the trees."
)

encoded = tokenizer(
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
)

available_tokens = encoded["input_ids"].shape[1]

print("Original prompt token count:", available_tokens)

Original prompt token count: 39


In [7]:
if available_tokens < PROMPT_LENGTH:
    raise ValueError(
        f"Prompt contains only {available_tokens} tokens. "
        f"At least {PROMPT_LENGTH} are required."
    )

input_ids = encoded["input_ids"][:, :PROMPT_LENGTH]
attention_mask = torch.ones_like(input_ids)

print("Benchmark input shape:", input_ids.shape)
print("Benchmark token count:", input_ids.shape[1])
print("Token IDs:", input_ids[0].tolist())
print("Truncated prompt:", tokenizer.decode(input_ids[0]))

assert input_ids.shape == (1, 32)

Benchmark input shape: torch.Size([1, 32])
Benchmark token count: 32
Token IDs: [7454, 2402, 257, 640, 11, 257, 11040, 1310, 2576, 3706, 20037, 5615, 287, 257, 5897, 7404, 13970, 257, 4077, 8222, 13, 3887, 3329, 11, 673, 5281, 4713, 8509, 284, 607, 18410, 290]
Truncated prompt: Once upon a time, a curious little girl named Lily lived in a quiet village beside a green forest. Every morning, she carried fresh bread to her grandmother and


In [8]:
block_input_lengths = []

def record_block_input_length(module, args):
    hidden_states = args[0]

    block_input_lengths.append(
        hidden_states.shape[1]
    )

hook_handle = (
    model.transformer.h[0]
    .register_forward_pre_hook(
        record_block_input_length
    )
)

In [9]:
with torch.no_grad():
    generated_sequences = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,

        min_new_tokens=GENERATED_LENGTH,
        max_new_tokens=GENERATED_LENGTH,

        do_sample=False,
        use_cache=True,

        pad_token_id=tokenizer.eos_token_id,
    )

hook_handle.remove()

In [10]:
generated_token_ids = generated_sequences[
    :,
    PROMPT_LENGTH:
]

print("Prompt tokens:", input_ids.shape[1])
print("Generated tokens:", generated_token_ids.shape[1])
print("Total tokens:", generated_sequences.shape[1])

assert input_ids.shape[1] == 32
assert generated_token_ids.shape[1] == 20
assert generated_sequences.shape[1] == 52

Prompt tokens: 32
Generated tokens: 20
Total tokens: 52


In [11]:
print(
    "Generated continuation:",
    tokenizer.decode(
        generated_token_ids[0],
        skip_special_tokens=True,
    ),
)

print(
    "Complete sequence:",
    tokenizer.decode(
        generated_sequences[0],
        skip_special_tokens=True,
    ),
)

Generated continuation:  asked her to help her.

One day, Lily's grandmother came to visit and asked her
Complete sequence: Once upon a time, a curious little girl named Lily lived in a quiet village beside a green forest. Every morning, she carried fresh bread to her grandmother and asked her to help her.

One day, Lily's grandmother came to visit and asked her


In [12]:
assert block_input_lengths[0] == 32
assert len(block_input_lengths) == 20
assert all(
    length == 1
    for length in block_input_lengths[1:]
)

print("KV-cache behavior verified")

KV-cache behavior verified


In [13]:
def create_block0_input(model, input_ids):
    batch_size, sequence_length = input_ids.shape

    position_ids = torch.arange(
        sequence_length,
        dtype=torch.long,
        device=input_ids.device,
    ).unsqueeze(0)

    with torch.no_grad():
        token_embeddings = model.transformer.wte(
            input_ids
        )

        position_embeddings = model.transformer.wpe(
            position_ids
        )

        block_input = model.transformer.drop(
            token_embeddings + position_embeddings
        )

    return block_input

In [14]:
block_input = create_block0_input(
    model,
    input_ids,
)

In [15]:
block_input.shape

torch.Size([1, 32, 128])

In [16]:
print("Token ID shape:", input_ids.shape)
print("Block input shape:", block_input.shape)
print("Block input dtype:", block_input.dtype)

print(
    "Block input range:",
    block_input.min().item(),
    block_input.max().item(),
)

Token ID shape: torch.Size([1, 32])
Block input shape: torch.Size([1, 32, 128])
Block input dtype: torch.float32
Block input range: -0.3287893235683441 0.30728602409362793


In [17]:
from torch import nn

In [18]:
class BlockWrapper(nn.Module):
    def __init__(self, transformer_block):
        super().__init__()

        self.transformer_block = transformer_block

    def forward(self, hidden_states):
        block_outputs = self.transformer_block(
            hidden_states,
            layer_past=None,
            attention_mask=None,
            use_cache=False,
            output_attentions=False,
        )

        output_hidden_states = block_outputs[0]

        return output_hidden_states

In [19]:
block0 = BlockWrapper(
    model.transformer.h[0]
)

block0.eval()

BlockWrapper(
  (transformer_block): GPTNeoBlock(
    (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (attn): GPTNeoAttention(
      (attention): GPTNeoSelfAttention(
        (attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_dropout): Dropout(p=0.0, inplace=False)
        (k_proj): Linear(in_features=128, out_features=128, bias=False)
        (v_proj): Linear(in_features=128, out_features=128, bias=False)
        (q_proj): Linear(in_features=128, out_features=128, bias=False)
        (out_proj): Linear(in_features=128, out_features=128, bias=True)
      )
    )
    (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (mlp): GPTNeoMLP(
      (c_fc): Linear(in_features=128, out_features=512, bias=True)
      (c_proj): Linear(in_features=512, out_features=128, bias=True)
      (act): NewGELUActivation()
      (dropout): Dropout(p=0.0, inplace=False)
    )
  )
)

In [20]:
with torch.no_grad():
    block_output_fp32 = block0(
        block_input
    )

In [21]:
print("Block input shape:",
      block_input.shape)

print("Block output shape:",
      block_output_fp32.shape)

print("Input dtype:",
      block_input.dtype)

print("Output dtype:",
      block_output_fp32.dtype)

print(
    "Output range:",
    block_output_fp32.min().item(),
    block_output_fp32.max().item(),
)

Block input shape: torch.Size([1, 32, 128])
Block output shape: torch.Size([1, 32, 128])
Input dtype: torch.float32
Output dtype: torch.float32
Output range: -0.6481602787971497 0.5356095433235168


In [22]:
def export_block(
    block,
    example_input,
):
    block.eval()

    example_arguments = (
        example_input.detach(),
    )
    print(type(example_input))
    print(type(example_arguments))

    exported_program = torch.export.export(
        block,
        example_arguments,
    )

    return exported_program

In [23]:
exported_block0 = export_block(
    block0,
    block_input,
)

<class 'torch.Tensor'>
<class 'tuple'>


In [24]:
print(type(exported_block0))
print(
    "Graph node count:",
    len(list(exported_block0.graph.nodes)),
)

<class 'torch.export.exported_program.ExportedProgram'>
Graph node count: 62


In [25]:
exported_block0_module = (
    exported_block0.module()
)

In [26]:
with torch.no_grad():
    exported_output = (
        exported_block0_module(
            block_input
        )
    )

In [27]:
maximum_difference = (
    block_output_fp32
    - exported_output
).abs().max().item()

print(
    "Maximum original/exported difference:",
    maximum_difference,
)

print(
    "Exported output shape:",
    exported_output.shape,
)

Maximum original/exported difference: 0.0
Exported output shape: torch.Size([1, 32, 128])


In [28]:
from litert_torch.quantize.pt2e_quantizer import (
    PT2EQuantizer,
    get_symmetric_quantization_config,
)

def create_w8a8_quantizer():
    quantization_config = get_symmetric_quantization_config(
        is_per_channel=False,
        is_dynamic=False,
        is_qat=False,
    )

    quantizer = PT2EQuantizer().set_global(
        quantization_config
    )

    return quantizer


quantizer = create_w8a8_quantizer()
print(quantizer)

I0000 00:00:1790171311.037874   23159 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [29]:
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e

prepared_block0 = prepare_pt2e(
    exported_block0.module(),
    quantizer,
)

print(prepared_block0)

GraphModule(
  (transformer_block): Module(
    (ln_1): Module()
    (attn): Module(
      (attention): Module(
        (k_proj): Module()
        (v_proj): Module()
        (q_proj): Module()
        (out_proj): Module()
      )
    )
    (ln_2): Module()
    (mlp): Module(
      (c_fc): Module()
      (c_proj): Module()
    )
  )
  (activation_post_process_3): MinMaxObserver(min_val=inf, max_val=-inf)
  (activation_post_process_5): MinMaxObserver(min_val=inf, max_val=-inf)
  (activation_post_process_1): MinMaxObserver(min_val=inf, max_val=-inf)
  (activation_post_process_14): MinMaxObserver(min_val=inf, max_val=-inf)
  (activation_post_process_17): MinMaxObserver(min_val=inf, max_val=-inf)
  (activation_post_process_29): MinMaxObserver(min_val=inf, max_val=-inf)
  (_guards_fn): GuardsFn()
  (activation_post_process_0): HistogramObserver(min_val=inf, max_val=-inf)
  (activation_post_process_2): HistogramObserver(min_val=inf, max_val=-inf)
  (activation_post_process_4): HistogramObserv

In [30]:
def calibrate_block(prepared_block, calibration_input):
    with torch.no_grad():
        output = prepared_block(calibration_input)

    return output


calibration_output = calibrate_block(
    prepared_block0,
    block_input,
)

In [31]:
observer = prepared_block0.activation_post_process_0

print("Minimum:", observer.min_val)
print("Maximum:", observer.max_val)
print("Output shape:", calibration_output.shape)

Minimum: tensor(-3.7485)
Maximum: tensor(3.0682)
Output shape: torch.Size([1, 32, 128])


In [32]:
from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e

quantized_block0 = convert_pt2e(prepared_block0)

print(quantized_block0)

GraphModule(
  (transformer_block): Module(
    (ln_1): Module()
    (attn): Module(
      (attention): Module(
        (k_proj): Module()
        (v_proj): Module()
        (q_proj): Module()
        (out_proj): Module()
      )
    )
    (ln_2): Module()
    (mlp): Module(
      (c_fc): Module()
      (c_proj): Module()
    )
  )
  (_guards_fn): GuardsFn()
)



def forward(self, hidden_states):
    hidden_states, = fx_pytree.tree_flatten_spec(([hidden_states], {}), self._in_spec)
    transformer_block_ln_1_weight = self.transformer_block.ln_1.weight
    transformer_block_ln_1_bias = self.transformer_block.ln_1.bias
    quantize_per_tensor_default = self._frozen_param0
    dequantize_per_tensor_default = torch.ops.quantized_decomposed.dequantize_per_tensor.default(quantize_per_tensor_default, 0.0017351132119074464, 0, -127, 127, torch.int8);  quantize_per_tensor_default = None
    quantize_per_tensor_default_1 = self._frozen_param1
    dequantize_per_tensor_default_1 = torch.ops.quant

In [33]:
with torch.no_grad():
    quantized_output = quantized_block0(block_input)

print("Output shape:", quantized_output.shape)
print("Output dtype:", quantized_output.dtype)

Output shape: torch.Size([1, 32, 128])
Output dtype: torch.float32


In [34]:
absolute_error = (
    block_output_fp32 - quantized_output
).abs()

print("Maximum absolute error:", absolute_error.max().item())
print("Mean absolute error:", absolute_error.mean().item())

Maximum absolute error: 0.007108405232429504
Mean absolute error: 0.001388444914482534


In [35]:
def compare_outputs(test_block_input):
    with torch.no_grad():
        fp32_output = block0(test_block_input)
        int8_output = quantized_block0(test_block_input)

    error = (fp32_output - int8_output).abs()

    print("Maximum absolute error:",
          error.max().item())

    print("Mean absolute error:",
          error.mean().item())

In [36]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [37]:
new_prompt = "The little dog walked into the forest"

encoded = tokenizer(
    new_prompt,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=32,
)

new_input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

print("Input shape:", new_input_ids.shape)
print("Real token count:", attention_mask.sum().item())
print("Pad token:", tokenizer.pad_token)
print("Pad token ID:", tokenizer.pad_token_id)

Input shape: torch.Size([1, 32])
Real token count: 7
Pad token: <|endoftext|>
Pad token ID: 50256


In [38]:
sequence_length = new_input_ids.shape[1]

new_position_ids = torch.arange(
    sequence_length,
    device=new_input_ids.device,
).unsqueeze(0)

with torch.no_grad():
    new_block_input = (
        model.transformer.wte(new_input_ids)
        + model.transformer.wpe(new_position_ids)
    )

In [39]:
with torch.no_grad():
    fp32_output = block0(new_block_input)
    quantized_output = quantized_block0(new_block_input)

absolute_error = (
    fp32_output - quantized_output
).abs()

print("Maximum absolute error:",
      absolute_error.max().item())

print("Mean absolute error:",
      absolute_error.mean().item())

Maximum absolute error: 0.03312571346759796
Mean absolute error: 0.0016461079940199852


In [40]:
real_token_mask = (
    attention_mask
    .unsqueeze(-1)
    .expand_as(absolute_error)
    .bool()
)

real_token_error = absolute_error[real_token_mask]

print(
    "Real-token maximum error:",
    real_token_error.max().item(),
)

print(
    "Real-token mean error:",
    real_token_error.mean().item(),
)

Real-token maximum error: 0.0258101224899292
Real-token mean error: 0.0015851309290155768


In [41]:
fresh_exported_block0 = export_block(
    block0,
    block_input,
)

fresh_quantizer = create_w8a8_quantizer()

prepared_block0_multi = prepare_pt2e(
    fresh_exported_block0.module(),
    fresh_quantizer,
)

<class 'torch.Tensor'>
<class 'tuple'>


In [42]:
calibration_prompts = [
    """
    Once upon a time, a little girl named Lily lived beside a quiet
    forest. Every morning she carried bread to her grandmother and
    greeted all the animals along the path.
    """,

    """
    Tom found a small red ball underneath his bed. He took it outside
    and played with his dog until the sun disappeared behind the hills.
    """,

    """
    A young rabbit wanted to cross the river, but the water was too
    deep. His friends gathered branches and helped him build a bridge.
    """,

    """
    Mia planted a tiny seed in the garden. She watered it every day,
    and soon a tall sunflower grew beside her bedroom window.
    """,

    """
    The little bird could not fly as high as the others. She practiced
    every afternoon until she finally reached the top of the old tree.
    """,

    """
    Ben heard a strange sound inside the kitchen. He opened the door
    slowly and discovered a hungry kitten looking for some food.
    """,

    """
    A kind farmer shared his vegetables with the people in his village.
    When winter arrived, everyone helped him repair his wooden barn.
    """,

    """
    Sara lost her favorite blue hat at the park. She searched near the
    swings and found it beside a sleeping brown puppy.
    """,
]

In [43]:
all_calibration_ids = []

for prompt in calibration_prompts:
    token_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
    )

    all_calibration_ids.extend(token_ids)
    all_calibration_ids.append(tokenizer.eos_token_id)


calibration_chunks = []

for start in range(
    0,
    len(all_calibration_ids) - 31,
    32,
):
    chunk = all_calibration_ids[
        start : start + 32
    ]

    calibration_chunks.append(chunk)


print("Number of calibration chunks:",
      len(calibration_chunks))

print("Tokens in each chunk:",
      len(calibration_chunks[0]))

Number of calibration chunks: 10
Tokens in each chunk: 32


In [44]:
position_ids_32 = torch.arange(
    32,
).unsqueeze(0)

with torch.no_grad():
    for chunk in calibration_chunks:
        calibration_ids = torch.tensor(
            [chunk],
            dtype=torch.long,
        )

        calibration_block_input = (
            model.transformer.wte(calibration_ids)
            + model.transformer.wpe(position_ids_32)
        )

        prepared_block0_multi(
            calibration_block_input
        )

In [45]:
observer = (
    prepared_block0_multi
    .activation_post_process_0
)

print("Observed minimum:", observer.min_val)
print("Observed maximum:", observer.max_val)

Observed minimum: tensor(-3.7441)
Observed maximum: tensor(4.0328)


In [46]:
quantized_block0_multi = convert_pt2e(
    prepared_block0_multi
)

In [47]:
with torch.no_grad():
    fp32_test_output = block0(
        new_block_input
    )

    quantized_test_output = (
        quantized_block0_multi(
            new_block_input
        )
    )

In [48]:
multi_calibration_error = (
    fp32_test_output
    - quantized_test_output
).abs()

print(
    "Output shape:",
    quantized_test_output.shape,
)

print(
    "Output dtype:",
    quantized_test_output.dtype,
)

print(
    "Maximum absolute error:",
    multi_calibration_error.max().item(),
)

print(
    "Mean absolute error:",
    multi_calibration_error.mean().item(),
)

Output shape: torch.Size([1, 32, 128])
Output dtype: torch.float32
Maximum absolute error: 0.007028862833976746
Mean absolute error: 0.0014894487103447318


In [49]:
real_token_mask = (
    attention_mask
    .unsqueeze(-1)
    .expand_as(multi_calibration_error)
    .bool()
)

real_token_error_multi = (
    multi_calibration_error[
        real_token_mask
    ]
)

print(
    "Real-token maximum error:",
    real_token_error_multi.max().item(),
)

print(
    "Real-token mean error:",
    real_token_error_multi.mean().item(),
)

Real-token maximum error: 0.007028862833976746
Real-token mean error: 0.0015181024791672826


In [51]:
litert_exported_block0 = export_block(
    block0,
    block_input,
)

litert_quantizer = create_w8a8_quantizer()

litert_prepared_block0 = prepare_pt2e(
    litert_exported_block0.module(),
    litert_quantizer,
)

position_ids_32 = torch.arange(32).unsqueeze(0)

with torch.no_grad():
    for chunk in calibration_chunks:
        calibration_ids = torch.tensor(
            [chunk],
            dtype=torch.long,
        )

        calibration_block_input = (
            model.transformer.wte(calibration_ids)
            + model.transformer.wpe(position_ids_32)
        )

        litert_prepared_block0(
            calibration_block_input
        )

<class 'torch.Tensor'>
<class 'tuple'>


In [52]:
litert_quantized_block0 = convert_pt2e(
    litert_prepared_block0,
    fold_quantize=False,
)

In [53]:
import litert_torch

from litert_torch.quantize.quant_config import (
    QuantConfig,
)

sample_args = (
    block_input.detach(),
)

edge_block0 = litert_torch.convert(
    litert_quantized_block0,
    sample_args,
    quant_config=QuantConfig(
        pt2e_quantizer=litert_quantizer
    ),
)

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:00)

(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default (+00:00)

... (14 outputs left out)

(00:01) [ DONE] LiteRT-Torch Convert > Merge MLIR Modules (+00:00)

(00:01) [START] LiteRT-Torch Convert > Run LiteRT Converter Passes

(00:01) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:01) [ DONE] LiteRT-Torch Convert (+00:01)

In [54]:
import numpy as np
import torch

with torch.no_grad():
    fp32_output = block0(
        new_block_input
    )

    pt2e_output = litert_quantized_block0(
        new_block_input
    )

litert_output = edge_block0(
    new_block_input.detach()
)

(00:00) [START] Write Model to Bytes

I0000 00:00:1790171523.252975   23159 flatbuffer_export.cc:4342] Estimated count of arithmetic ops: 14.844 M  ops, equivalently 7.422 M  MACs


(00:00) [ DONE] Write Model to Bytes (+00:00)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
ERROR: Subgraph reshaping is enabled by default, TFLITE_XNNPACK_DELEGATE_FLAG_ENABLE_SUBGRAPH_RESHAPING is deprecated and will be removed in the future.


In [55]:
if isinstance(litert_output, (list, tuple)):
    litert_output = litert_output[0]

fp32_numpy = (
    fp32_output.detach().cpu().numpy()
)

pt2e_numpy = (
    pt2e_output.detach().cpu().numpy()
)

litert_numpy = np.asarray(litert_output)

print("FP32 shape:", fp32_numpy.shape)
print("PT2E shape:", pt2e_numpy.shape)
print("LiteRT shape:", litert_numpy.shape)
print("LiteRT dtype:", litert_numpy.dtype)

FP32 shape: (1, 32, 128)
PT2E shape: (1, 32, 128)
LiteRT shape: (1, 32, 128)
LiteRT dtype: float32


In [56]:
conversion_error = np.abs(
    pt2e_numpy - litert_numpy
)

print(
    "PT2E/LiteRT maximum error:",
    conversion_error.max(),
)

print(
    "PT2E/LiteRT mean error:",
    conversion_error.mean(),
)

print(
    "PT2E/LiteRT close:",
    np.allclose(
        pt2e_numpy,
        litert_numpy,
        atol=1e-4,
        rtol=1e-4,
    ),
)

PT2E/LiteRT maximum error: 0.0026784912
PT2E/LiteRT mean error: 1.7656032e-05
PT2E/LiteRT close: False


In [57]:
fp32_litert_error = np.abs(
    fp32_numpy - litert_numpy
)

print(
    "FP32/LiteRT maximum error:",
    fp32_litert_error.max(),
)

print(
    "FP32/LiteRT mean error:",
    fp32_litert_error.mean(),
)

FP32/LiteRT maximum error: 0.007028863
FP32/LiteRT mean error: 0.0014921346


In [58]:
real_token_mask_numpy = np.broadcast_to(
    attention_mask
    .cpu()
    .numpy()
    .astype(bool)[..., None],
    fp32_litert_error.shape,
)

real_token_litert_error = (
    fp32_litert_error[
        real_token_mask_numpy
    ]
)

print(
    "Real-token FP32/LiteRT maximum:",
    real_token_litert_error.max(),
)

print(
    "Real-token FP32/LiteRT mean:",
    real_token_litert_error.mean(),
)

Real-token FP32/LiteRT maximum: 0.007028863
Real-token FP32/LiteRT mean: 0.0015170646


In [59]:
import os

tflite_path = "block0_w8a8.tflite"

edge_block0.export(tflite_path)

print(
    "File exists:",
    os.path.isfile(tflite_path),
)

print(
    "File size:",
    os.path.getsize(tflite_path),
    "bytes",
)

File exists: True
File size: 233360 bytes


In [60]:
reloaded_block0 = litert_torch.load(
    tflite_path
)

reloaded_output = reloaded_block0(
    new_block_input.detach()
)

if isinstance(reloaded_output, (list, tuple)):
    reloaded_output = reloaded_output[0]

reloaded_numpy = np.asarray(
    reloaded_output
)

reload_error = np.abs(
    litert_numpy - reloaded_numpy
)

print(
    "Original/reloaded maximum error:",
    reload_error.max(),
)

print(
    "Original/reloaded mean error:",
    reload_error.mean(),
)

Original/reloaded maximum error: 0.0
Original/reloaded mean error: 0.0


In [62]:
from ai_edge_litert.interpreter import Interpreter
import numpy as np
from collections import Counter

interpreter = Interpreter(
    model_path="block0_w8a8.tflite",
    experimental_preserve_all_tensors=True,
)

interpreter.allocate_tensors()

tensor_details = interpreter.get_tensor_details()

tensor_by_index = {
    tensor["index"]: tensor
    for tensor in tensor_details
}

/home/randy/Documents/ACompArc/model/.venv/lib/python3.12/site-packages/ai_edge_litert/interpreter.py:480: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(


In [63]:
dtype_counts = Counter(
    np.dtype(tensor["dtype"]).name
    for tensor in tensor_details
)

print("Tensor dtype counts:")

for dtype, count in dtype_counts.items():
    print(dtype, count)

Tensor dtype counts:
float32 56
int32 14
bool 1
int8 37


In [64]:
matrix_operator_names = {
    "FULLY_CONNECTED",
    "BATCH_MATMUL",
}

w8a8_operator_count = 0

for operator_number, operator in enumerate(
    interpreter._get_ops_details()
):
    operator_name = operator["op_name"]

    if operator_name not in matrix_operator_names:
        continue

    input_tensors = [
        tensor_by_index[int(index)]
        for index in operator["inputs"]
        if int(index) >= 0
    ]

    output_tensors = [
        tensor_by_index[int(index)]
        for index in operator["outputs"]
        if int(index) >= 0
    ]

    print(
        f"\nOperator {operator_number}:",
        operator_name,
    )

    for position, tensor in enumerate(input_tensors):
        qparams = tensor["quantization_parameters"]

        print(
            f"  Input {position}:",
            "dtype =",
            np.dtype(tensor["dtype"]).name,
            "shape =",
            tensor["shape"].tolist(),
            "scale count =",
            len(qparams["scales"]),
            "name =",
            tensor["name"],
        )

    for position, tensor in enumerate(output_tensors):
        qparams = tensor["quantization_parameters"]

        print(
            f"  Output {position}:",
            "dtype =",
            np.dtype(tensor["dtype"]).name,
            "shape =",
            tensor["shape"].tolist(),
            "scale count =",
            len(qparams["scales"]),
        )

    if (
        len(input_tensors) >= 2
        and input_tensors[0]["dtype"] == np.int8
        and input_tensors[1]["dtype"] == np.int8
    ):
        w8a8_operator_count += 1
        print("  Classification: W8A8")
    else:
        print("  Classification: not W8A8")

print(
    "\nNumber of W8A8 matrix operators:",
    w8a8_operator_count,
)


Operator 15: FULLY_CONNECTED
  Input 0: dtype = int8 shape = [1, 32, 128] scale count = 1 name = torch.export._trace._non_strict_export.<locals>._tuplify_outputs.<locals>._aot_export_non_strict.<locals>.Wrapper;6
  Input 1: dtype = int8 shape = [128, 128] scale count = 128 name = torch.export._trace._non_strict_export.<locals>._tuplify_outputs.<locals>._aot_export_non_strict.<locals>.Wrapper;2
  Output 0: dtype = int8 shape = [1, 32, 128] scale count = 1
  Classification: W8A8

Operator 16: FULLY_CONNECTED
  Input 0: dtype = int8 shape = [1, 32, 128] scale count = 1 name = torch.export._trace._non_strict_export.<locals>._tuplify_outputs.<locals>._aot_export_non_strict.<locals>.Wrapper;6
  Input 1: dtype = int8 shape = [128, 128] scale count = 128 name = torch.export._trace._non_strict_export.<locals>._tuplify_outputs.<locals>._aot_export_non_strict.<locals>.Wrapper;
  Output 0: dtype = int8 shape = [1, 32, 128] scale count = 1
  Classification: W8A8

Operator 17: FULLY_CONNECTED
  Inp

In [65]:
for tensor in tensor_details:
    if tensor["dtype"] == np.float32:
        print(
            "shape =",
            tensor["shape"].tolist(),
            "name =",
            tensor["name"],
        )

shape = [1, 32, 128] name = serving_default_args_0
shape = [] name = arith.constant
shape = [] name = __main__.BlockWrapper/transformers.models.gpt_neo.modeling_gpt_neo.GPTNeoBlock_transformer_block/transformers.models.gpt_neo.modeling_gpt_neo.GPTNeoMLP_mlp/transformers.activations.NewGELUActivation_act;
shape = [] name = __main__.BlockWrapper/transformers.models.gpt_neo.modeling_gpt_neo.GPTNeoBlock_transformer_block/transformers.models.gpt_neo.modeling_gpt_neo.GPTNeoMLP_mlp/transformers.activations.NewGELUActivation_act;1
shape = [] name = arith.constant1
shape = [128] name = arith.constant2
shape = [128] name = arith.constant3
shape = [128] name = arith.constant4
shape = [128] name = arith.constant5
shape = [] name = __main__.BlockWrapper/transformers.models.gpt_neo.modeling_gpt_neo.GPTNeoBlock_transformer_block/transformers.models.gpt_neo.modeling_gpt_neo.GPTNeoAttention_attn/transformers.models.gpt_neo.modeling_gpt_neo.GPTNeoSelfAttention_attention;2
shape = [32] name = __main__.Bl

In [66]:
import numpy as np

matrix_operations = {
    "FULLY_CONNECTED",
    "BATCH_MATMUL",
}

matrix_number = 0

for operator_index, operator in enumerate(
    interpreter._get_ops_details()
):
    if operator["op_name"] not in matrix_operations:
        continue

    matrix_number += 1

    print(
        "\n================================"
    )
    print("Matrix operation:", matrix_number)
    print("Graph operator index:", operator_index)
    print("Operator:", operator["op_name"])

    print("\nInputs:")

    for input_position, tensor_index in enumerate(
        operator["inputs"]
    ):
        tensor_index = int(tensor_index)

        if tensor_index < 0:
            continue

        tensor = tensor_by_index[tensor_index]
        qparams = tensor["quantization_parameters"]

        print(
            f"  Input {input_position}"
        )
        print("    tensor index:",
              tensor_index)
        print("    name:",
              tensor["name"])
        print("    shape:",
              tensor["shape"].tolist())
        print("    dtype:",
              np.dtype(tensor["dtype"]).name)
        print("    scales:",
              qparams["scales"])
        print("    zero points:",
              qparams["zero_points"])

    print("\nOutputs:")

    for output_position, tensor_index in enumerate(
        operator["outputs"]
    ):
        tensor_index = int(tensor_index)
        tensor = tensor_by_index[tensor_index]
        qparams = tensor["quantization_parameters"]

        print(
            f"  Output {output_position}"
        )
        print("    tensor index:",
              tensor_index)
        print("    name:",
              tensor["name"])
        print("    shape:",
              tensor["shape"].tolist())
        print("    dtype:",
              np.dtype(tensor["dtype"]).name)
        print("    scales:",
              qparams["scales"])
        print("    zero points:",
              qparams["zero_points"])


Matrix operation: 1
Graph operator index: 15
Operator: FULLY_CONNECTED

Inputs:
  Input 0
    tensor index: 45
    name: torch.export._trace._non_strict_export.<locals>._tuplify_outputs.<locals>._aot_export_non_strict.<locals>.Wrapper;6
    shape: [1, 32, 128]
    dtype: int8
    scales: [0.03048289]
    zero points: [-5]
  Input 1
    tensor index: 27
    name: torch.export._trace._non_strict_export.<locals>._tuplify_outputs.<locals>._aot_export_non_strict.<locals>.Wrapper;2
    shape: [128, 128]
    dtype: int8
    scales: [0.00173675 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675
 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675
 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675
 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675
 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675
 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675
 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675 0.00173675

In [67]:
import numpy as np

model_input = interpreter.get_input_details()[0]

test_input_numpy = (
    new_block_input
    .detach()
    .cpu()
    .numpy()
    .astype(np.float32)
)

interpreter.set_tensor(
    model_input["index"],
    test_input_numpy,
)

interpreter.invoke()

In [68]:
q_proj_input_int8 = (
    interpreter.get_tensor(45).copy()
)

q_proj_weight_int8 = (
    interpreter.get_tensor(27).copy()
)

q_proj_output_int8 = (
    interpreter.get_tensor(46).copy()
)

In [69]:
print(
    "Input:",
    q_proj_input_int8.shape,
    q_proj_input_int8.dtype,
    q_proj_input_int8.min(),
    q_proj_input_int8.max(),
)

print(
    "Weight:",
    q_proj_weight_int8.shape,
    q_proj_weight_int8.dtype,
    q_proj_weight_int8.min(),
    q_proj_weight_int8.max(),
)

print(
    "Output:",
    q_proj_output_int8.shape,
    q_proj_output_int8.dtype,
    q_proj_output_int8.min(),
    q_proj_output_int8.max(),
)

print(
    "\nFirst input vector:",
    q_proj_input_int8[0, 0, :16],
)

print(
    "\nFirst weight row:",
    q_proj_weight_int8[0, :16],
)

print(
    "\nFirst output values:",
    q_proj_output_int8[0, 0, :16],
)

Input: (1, 32, 128) int8 -93 101
Weight: (128, 128) int8 -116 127
Output: (1, 32, 128) int8 -104 58

First input vector: [-36 -22   8  -4 -12  15 -38 -30  47 -63  25  23  27  48  -2  42]

First weight row: [  6  -4   7  -6  -6   4  17  13  -4  -9  -1  15 -20  -2  -4  -5]

First output values: [-13 -22 -18  -7 -19 -14 -27 -30 -20 -14 -16 -29 -35  -2 -26 -25]
